In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [3]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))#Scaling of each channel (R,G,B)
])

trainset=CIFAR10(root='./data',train=True,download=True,transform=transform)
testset=CIFAR10(root='./data',train=False,download=True,transform=transform)

trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)


Files already downloaded and verified
Files already downloaded and verified


In [9]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),#32 is the feature maps( 1 feature map=1 kernel ,like that 32 kernels)
        nn.ReLU(),
        nn.MaxPool2d(2,2),#Kernel size 2,stride=2

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x
            
        

In [10]:
model=CNN()

In [11]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [17]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    for images,labels in trainloader:
        optimizer.zero_grad()

        output=model.forward(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss+=loss.item()

    print(f"{epoch+1}/{epochs} and loss={epoch_training_loss/len(trainloader)}")
        

1/10 and loss=0.05555679430832158
2/10 and loss=0.04813168283169227
3/10 and loss=0.047738274301872996
4/10 and loss=0.04911992024071157
5/10 and loss=0.04491325420897801
6/10 and loss=0.04923989028835919
7/10 and loss=0.040367071683833224
8/10 and loss=0.05421044012989819
9/10 and loss=0.04660571441091747
10/10 and loss=0.04882369822189968


In [18]:
correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)

        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)
print(f"Accuracy={correct_labels/total_labels*100}")


Accuracy=75.06
